In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

In [24]:
print("Loading dataset in Colab...")

df = pd.read_csv("final_blood_donation_dataset4new 2.csv", skiprows=1, on_bad_lines='skip')


df.columns = ['Recency', 'Frequency', 'Monetary', 'Time', 'Donated', 'Blood_Group', 'Age', 'City']

numeric_cols = ['Recency', 'Frequency', 'Monetary', 'Time', 'Age']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Donated'] = pd.to_numeric(df['Donated'], errors='coerce').fillna(0).astype(int)
df = df.dropna()

print("Dataset Shape:", df.shape)
df.head()

Loading dataset in Colab...
Dataset Shape: (748, 8)


,Recency,Frequency,Monetary,Time,Donated,Blood_Group,Age,City
0,2,50,12500,98,1,B-,43,Mumbai
1,0,13,3250,28,1,A-,47,Mumbai
2,1,16,4000,35,1,A+,55,Mumbai
3,2,20,5000,45,1,B+,21,Bangalore
4,1,24,6000,77,0,B-,39,Bangalore


In [25]:
X = df[['Recency', 'Frequency', 'Monetary', 'Time', 'Age', 'Blood_Group', 'City']]
y = df['Donated']

# 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Smart Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Recency', 'Frequency', 'Monetary', 'Time', 'Age']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Blood_Group', 'City'])
    ])

In [26]:
# Yeh woh settings hain jisse accuracy 80% cross karegi
print("Setting up High-Accuracy Models...")
model1 = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model2 = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
model3 = LogisticRegression(max_iter=1000, random_state=42)
model4 = KNeighborsClassifier(n_neighbors=11)
model5 = SVC(probability=True, kernel='rbf', C=1.0, random_state=42)

# 5-in-1 Ensemble Classifier
ensemble_model = VotingClassifier(
    estimators=[
        ('RandomForest', model1),
        ('GradientBoost', model2),
        ('LogisticReg', model3),
        ('KNN', model4),
        ('SVM', model5)
    ],
    voting='soft'
)

Setting up High-Accuracy Models...


In [31]:
# --- ADVANCED HYPER-TUNED MODELS (Targeting 85% Accuracy) ---
print("Setting up High-Accuracy Models...\n")

models = {
    # Trees badhaye, depth badhayi, aur overfitting rokne ke liye min_samples_split lagaya
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_split=5, random_state=42),

    # Learning slow ki aur trees badhaye taaki model better seekhe
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),

    # Standard Regression with specific Regularization (C=0.5)
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.5, random_state=42),

    # Neighbors 9 kiye aur distance-based weight lagaya
    "KNN": KNeighborsClassifier(n_neighbors=9, weights='distance'),

    # SVM ko thoda aur strict banaya (C=1.5)
    "SVC": SVC(probability=True, kernel='rbf', C=1.5, random_state=42)
}

# --- 1. ALAG-ALAG MODELS KI ACCURACY PRINT KARNA ---
print("--- Individual Model Accuracies ---")
for name, model in models.items():
    # Har model ke liye ek temporary pipeline banakar test karna
    temp_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    temp_pipeline.fit(X_train, y_train)
    y_pred_single = temp_pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred_single)
    print(f"{name} Accuracy: {acc * 100:.2f}%")


# --- 2. FINAL 5-IN-1 SUPER MODEL BANANA ---
print("\n--- Training Final 5-in-1 Super Model ---")
ensemble_model = VotingClassifier(
    estimators=[
        ('RandomForest', models["Random Forest"]),
        ('GradientBoost', models["Gradient Boosting"]),
        ('LogisticReg', models["Logistic Regression"]),
        ('KNN', models["KNN"]),
        ('SVM', models["SVC"])
    ],
    voting='soft' # Soft voting se sabhi models apni probabilities combine karte hain
)

final_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', ensemble_model)])
final_pipeline.fit(X_train, y_train)

# --- 3. FINAL RESULTS OVERALL ---
y_pred_final = final_pipeline.predict(X_test)

print("\n--- FINAL ENSEMBLE RESULTS ---")
print(f"OVERALL SUPER MODEL ACCURACY: {accuracy_score(y_test, y_pred_final) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred_final))

Setting up High-Accuracy Models...

--- Individual Model Accuracies ---
Random Forest Accuracy: 78.00%
Gradient Boosting Accuracy: 76.67%
Logistic Regression Accuracy: 78.00%
KNN Accuracy: 74.67%
SVC Accuracy: 77.33%

--- Training Final 5-in-1 Super Model ---

--- FINAL ENSEMBLE RESULTS ---
OVERALL SUPER MODEL ACCURACY: 78.00%

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.98      0.87       113
           1       0.75      0.16      0.27        37

    accuracy                           0.78       150
   macro avg       0.77      0.57      0.57       150
weighted avg       0.77      0.78      0.72       150

